# HR Assistant Agent - 프로그래밍 방식(코드 기반) Evaluator

이 Notebook에서는 Amazon Bedrock AgentCore Evaluations에서 **코드 기반 evaluator**를 생성하고 사용하는 방법을 보여 줍니다.

### 코드 기반 evaluator란?

사용자 지정 코드 기반 evaluator를 사용하면 Amazon Bedrock AgentCore Evaluations에서 **자체 Lambda 함수**를 evaluator로 사용할 수 있습니다.
이를 통해 다음과 같은 이점을 얻을 수 있습니다.

| 이점 | 설명 |
|---|---|
| **결정성** | LLM 편차 없이 동일한 입력에 항상 동일한 점수 생성 |
| **추론 비용 없음** | Lambda 실행 비용은 LLM 호출 비용의 일부에 불과함 |
| **도메인 특화** | 정확한 비즈니스 규칙 인코딩(예: "net pay must equal $X") |
| **조합 가능성** | 한 번의 실행에서 코드 기반 evaluator와 LLM 기반 evaluator 조합 |

### 구현할 항목

HR Assistant agent를 위한 Lambda 기반 evaluator 두 개를 생성합니다.

| Evaluator | 수준 | Lambda | 확인 항목 |
|---|---|---|---|
| **HRResponseLength** | TRACE | `hr-response-length` | 응답 길이가 50~600자인지 확인 |
| **HRFactChecker** | SESSION | `hr-fact-checker` | PTO 잔여 일수, 급여 명세서 및 정책 정보의 정확성 확인 |

그런 다음 코드 기반 evaluator와 기본 제공 LLM-as-a-Judge evaluator를 조합한 **혼합 evaluator 세트**로 `OnDemandEvaluationDatasetRunner`를 실행합니다. 실시간 모니터링을 위해 이러한 evaluator를 사용하는 온라인 평가도 설정합니다.

### 튜토리얼 세부 정보

| 항목 | 세부 정보 |
|---|---|
| Agent 프레임워크 | Strands Agents |
| Runtime | Amazon Bedrock AgentCore Runtime |
| Evaluation SDK | `bedrock-agentcore` ≥ 1.6.0 |
| AWS 서비스 | AgentCore Runtime, AgentCore Evaluations, Lambda, CloudWatch |

### 사전 요구 사항
- Python 3.10+
- AgentCore, Lambda, CloudWatch, ECR, IAM 권한이 있는 AWS 자격 증명
- 로컬에서 실행 중인 Docker(에이전트 컨테이너 이미지 빌드용)
- `AWSLambdaBasicExecutionRole`이 연결된 Lambda 실행용 IAM 역할


## 1단계: 종속성 설치

In [ ]:
!pip install -r requirements.txt -q

## 2단계: 구성

In [ ]:
import boto3
import io
import json
import os
import subprocess
import sys
import tempfile
import time
import uuid
import zipfile
from datetime import datetime
from pathlib import Path
from boto3.session import Session
from botocore.config import Config
from IPython.display import display, Markdown

# ── 리전 구성 ─────────────────────────────────────────────────────────────────
# REGION: AgentCore Runtime(에이전트)이 배포된 AWS 리전입니다.
#   boto3 세션에서 자동으로 감지합니다(AWS_DEFAULT_REGION 환경 변수 또는
#   ~/.aws/config의 기본 리전 읽기). 필요한 경우 다음과 같이 명시적으로 설정하세요.
#   REGION = "us-east-1"
#
#   groundtruth_evaluations.ipynb를 먼저 실행했다면 아래 agent-load 셀에서
#   %store로 REGION을 복원하며, 이 값을 재정의합니다.
REGION = Session().region_name
assert REGION, (
    "No AWS region detected. Set AWS_DEFAULT_REGION or configure a default "
    "region in ~/.aws/config, or set REGION explicitly above."
)

# EVAL_REGION: Lambda evaluator 및 evaluator 등록에 사용할 리전입니다.
#   온라인 평가에서는 반드시 REGION과 일치해야 합니다(에이전트의 CloudWatch 로그
#   그룹과 평가 구성이 같은 리전에 있어야 함). 아래 agent-clients 셀은 %store가
#   에이전트의 실제 리전을 복원한 후 EVAL_REGION을 REGION에 자동으로 맞춥니다.
EVAL_REGION = REGION

boto_session = Session(region_name=REGION)
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

# Lambda 실행 역할 - AWSLambdaBasicExecutionRole 필요
# 역할 이름이 다르면 이 값을 업데이트하세요
LAMBDA_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/AgentCoreLambdaExecutionRole"

print(f"Region          : {REGION}")
print(f"Eval Region     : {EVAL_REGION}")
print(f"Account         : {ACCOUNT_ID}")
print(f"Lambda Role ARN : {LAMBDA_ROLE_ARN}")

## 3단계: 에이전트 설정

AgentCore Runtime에 HR Assistant agent가 배포되어 있어야 합니다.
`groundtruth_evaluations.ipynb` Notebook을 실행했다면 에이전트 정보가 `%store`로 저장되어 있어
다시 불러올 수 있습니다. 그렇지 않으면 여기서 새로 배포합니다.

In [ ]:
# 정적 linter(ruff F821)가 미정의로 표시하지 않도록 미리 초기화합니다.
# %store -r은 이 값을 Ground Truth Notebook에서 저장한 값으로 덮어씁니다.
AGENT_ID = AGENT_ARN = CW_LOG_GROUP = ""

# Ground Truth Notebook에서 에이전트 정보를 불러옵니다.
# 현재 사용자의 boto3 기본값과 관계없이 agentcore 클라이언트가 올바른 배포 리전을
# 사용하도록 %store가 REGION을 복원합니다.
try:
    %store -r AGENT_ID
    %store -r AGENT_ARN
    %store -r CW_LOG_GROUP
    %store -r REGION          # 에이전트가 배포된 리전 복원
    print("Loaded agent from store:")
    print(f"  AGENT_ID     : {AGENT_ID}")
    print(f"  AGENT_ARN    : {AGENT_ARN}")
    print(f"  CW_LOG_GROUP : {CW_LOG_GROUP}")
    print(f"  REGION       : {REGION}")
    _agent_loaded = bool(AGENT_ID and AGENT_ARN)
except Exception:
    print("Agent info not found in store. Will deploy a fresh agent below.")
    _agent_loaded = False

In [ ]:
# 아직 불러오지 않았다면 에이전트 배포
if not _agent_loaded:
    # -------------------------------------------------------------------------
    # boto3(bedrock-agentcore-control) + Docker/ECR를 사용한 새 배포입니다.
    # 이 경로는 Ground Truth Notebook을 먼저 실행하지 않은 경우에만 실행됩니다.
    # CLI를 사용하려면 프로젝트 루트에서 `agentcore deploy`를 실행하고 아래의
    # AGENT_ID / AGENT_ARN / CW_LOG_GROUP를 수동으로 설정하세요.
    # -------------------------------------------------------------------------

    ecr_client = boto3.client("ecr", region_name=REGION)
    agentcore_control_deploy = boto3.client("bedrock-agentcore-control", region_name=REGION)
    iam_client = boto3.client("iam")

    AGENT_NAME = "hr_assistant_codeeval_tutorial"
    ECR_REPO_NAME = f"agentcore-{AGENT_NAME}"

    # ------------------------------------------------------------------
    # 1. IAM 실행 역할이 있는지 확인
    # ------------------------------------------------------------------
    EXECUTION_ROLE_NAME = "AgentCoreRuntimeExecutionRole"
    EXECUTION_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{EXECUTION_ROLE_NAME}"

    try:
        iam_client.get_role(RoleName=EXECUTION_ROLE_NAME)
        print(f"Using existing IAM role: {EXECUTION_ROLE_ARN}")
    except iam_client.exceptions.NoSuchEntityException:
        print(f"Creating IAM role: {EXECUTION_ROLE_NAME}...")
        trust_policy = json.dumps(
            {
                "Version": "2012-10-17",
                "Statement": [
                    {
                        "Effect": "Allow",
                        "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
                        "Action": "sts:AssumeRole",
                    }
                ],
            }
        )
        iam_client.create_role(
            RoleName=EXECUTION_ROLE_NAME,
            AssumeRolePolicyDocument=trust_policy,
            Description="Execution role for AgentCore Runtime tutorial agents",
        )
        for policy_arn in [
            "arn:aws:iam::aws:policy/AmazonBedrockFullAccess",
            "arn:aws:iam::aws:policy/CloudWatchLogsFullAccess",
        ]:
            iam_client.attach_role_policy(RoleName=EXECUTION_ROLE_NAME, PolicyArn=policy_arn)
        print(f"Created: {EXECUTION_ROLE_ARN}")
        time.sleep(10)  # IAM 전파 대기

    # ------------------------------------------------------------------
    # 2. ECR 리포지토리 생성(또는 기존 리포지토리 재사용)
    # ------------------------------------------------------------------
    try:
        ecr_resp = ecr_client.create_repository(repositoryName=ECR_REPO_NAME)
        ECR_REPO_URI = ecr_resp["repository"]["repositoryUri"]
        print(f"Created ECR repo: {ECR_REPO_URI}")
    except ecr_client.exceptions.RepositoryAlreadyExistsException:
        ECR_REPO_URI = ecr_client.describe_repositories(repositoryNames=[ECR_REPO_NAME])["repositories"][0][
            "repositoryUri"
        ]
        print(f"Using existing ECR repo: {ECR_REPO_URI}")

    IMAGE_URI = f"{ECR_REPO_URI}:latest"

    # ------------------------------------------------------------------
    # 3. Docker 이미지 빌드 및 ECR push
    # ------------------------------------------------------------------
    ecr_registry = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com"
    print("Docker login to ECR...")
    subprocess.run(
        f"aws ecr get-login-password --region {REGION} | docker login --username AWS --password-stdin {ecr_registry}",
        shell=True,
        check=True,
    )
    print("Building Docker image (this may take a few minutes)...")
    subprocess.run(
        ["docker", "build", "--platform", "linux/amd64", "-t", IMAGE_URI, "."],
        check=True,
    )
    print("Pushing image to ECR...")
    subprocess.run(["docker", "push", IMAGE_URI], check=True)
    print(f"Image pushed: {IMAGE_URI}")

    # AgentCore의 ECR pull 허용
    ecr_client.set_repository_policy(
        repositoryName=ECR_REPO_NAME,
        policyText=json.dumps(
            {
                "Version": "2012-10-17",
                "Statement": [
                    {
                        "Sid": "AllowAgentCorePull",
                        "Effect": "Allow",
                        "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
                        "Action": [
                            "ecr:GetDownloadUrlForLayer",
                            "ecr:BatchGetImage",
                            "ecr:BatchCheckLayerAvailability",
                        ],
                    }
                ],
            }
        ),
    )

    # ------------------------------------------------------------------
    # 4. AgentCore Runtime 생성(또는 업데이트)
    # ------------------------------------------------------------------
    artifact = {"containerConfiguration": {"containerUri": IMAGE_URI}}
    try:
        resp = agentcore_control_deploy.create_agent_runtime(
            agentRuntimeName=AGENT_NAME,
            agentRuntimeArtifact=artifact,
            executionRoleArn=EXECUTION_ROLE_ARN,
            networkConfiguration={"networkMode": "PUBLIC"},
        )
        AGENT_ID = resp["agentRuntimeId"]
        AGENT_ARN = resp["agentRuntimeArn"]
        print(f"Created AgentCore Runtime: {AGENT_ID}")
    except agentcore_control_deploy.exceptions.ConflictException:
        runtimes = agentcore_control_deploy.list_agent_runtimes()["agentRuntimes"]
        existing = next((r for r in runtimes if r["agentRuntimeName"] == AGENT_NAME), None)
        assert existing, f"Runtime {AGENT_NAME} not found after conflict"
        AGENT_ID = existing["agentRuntimeId"]
        AGENT_ARN = existing["agentRuntimeArn"]
        agentcore_control_deploy.update_agent_runtime(
            agentRuntimeId=AGENT_ID,
            agentRuntimeArtifact=artifact,
        )
        print(f"Updated existing runtime: {AGENT_ID}")

    # READY 상태까지 대기
    terminal = {"READY", "CREATE_FAILED", "UPDATE_FAILED"}
    while True:
        status = agentcore_control_deploy.get_agent_runtime(agentRuntimeId=AGENT_ID)["status"]
        print(f"  Status: {status}")
        if status in terminal:
            break
        time.sleep(15)

    assert status == "READY", f"Deployment failed: {status}"
    CW_LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{AGENT_ID}-DEFAULT"

    print("\nAgent deployed:")
    print(f"  AGENT_ID     : {AGENT_ID}")
    print(f"  AGENT_ARN    : {AGENT_ARN}")
    print(f"  CW_LOG_GROUP : {CW_LOG_GROUP}")
else:
    print("Using existing agent — skipping deployment.")

In [ ]:
# ARN에서 에이전트의 실제 리전을 가져옵니다(%store가 불완전해도 동작).
# ARN 형식: arn:aws:bedrock-agentcore:{region}:{account}:runtime/{id}
_arn_region = AGENT_ARN.split(":")[3] if AGENT_ARN else REGION
if _arn_region and _arn_region != REGION:
    print(f"Note: agent is in {_arn_region}, overriding REGION={REGION}")
    REGION = _arn_region

# Lambda evaluator, evaluator 등록 및 온라인 평가 구성이 모두 에이전트의 CloudWatch
# 로그 그룹과 같은 리전에 있도록 EVAL_REGION을 에이전트 리전에 맞춥니다. 온라인 평가에서는
# 로그 그룹과 evaluator가 control plane 구성과 같은 리전에 있어야 합니다.
if EVAL_REGION != REGION:
    print(f"Aligning EVAL_REGION: {EVAL_REGION} → {REGION} (must match agent region for online eval)")
    EVAL_REGION = REGION

# 에이전트 호출용 boto3 클라이언트 - 에이전트와 같은 리전에 있어야 함
agentcore_client = boto3.client("bedrock-agentcore", region_name=REGION)
print(f"REGION      : {REGION}")
print(f"EVAL_REGION : {EVAL_REGION}")

## 4단계: Lambda evaluator 함수 정의

각 코드 기반 evaluator는 에이전트 span을 받아 점수를 반환하는 Lambda 함수입니다.

### Lambda 입력 구조

AgentCore는 다음 이벤트를 Lambda에 전달합니다.

```json
{
  "evaluationInput": {
    "sessionSpans": [ ... ]   // 에이전트 세션의 OTel span
  },
  "evaluationTarget": {       // SESSION 수준에서는 null
    "traceIds": [ ... ]       // TRACE 수준에서 제공
  },
  "evaluationLevel": "TRACE" // 또는 "SESSION"
}
```

### Span 구조(Strands / AgentCore OTel)

```
span.name                             → 예: "invoke_agent", "llm_call"
span.attributes.gen_ai.operation.name → 도구 호출의 경우 "execute_tool"
span.attributes.gen_ai.tool.name      → 도구 호출 span의 도구 이름
span.span_events[*].body.output
      .messages[*].content.message   → 최종 에이전트 응답 텍스트
```

### Lambda 반환 형식

```json
{ "value": 1.0, "label": "PASS", "explanation": "..." }   // 성공
{ "errorCode": "NoResponseFound", "errorMessage": "..." }  // 오류
```

### Evaluator 1: HRResponseLength(TRACE 수준)

에이전트 응답 길이가 허용 범위(50~600자)인지 확인합니다.
50자보다 짧은 응답은 불완전할 가능성이 높고, 600자보다 긴 응답은 지나치게 상세할 수 있습니다.

**TRACE 수준**에서 작동하므로 개별 에이전트 응답을 각각 평가합니다.

In [ ]:
os.makedirs("lambdas/hr_response_length", exist_ok=True)

lambda_response_length_code = '# pylint: disable=duplicate-code\n"""\nHRResponseLength - Code-Based Evaluator(TRACE 수준)\n\nSDK 1.6의 @custom_code_based_evaluator() 데코레이터를 사용합니다.\n에이전트 응답의 문자 수가 MIN_LENGTH와 MAX_LENGTH 사이인지 검사합니다.\n측정 전에 thinking 블록(<thinking>...</thinking>)을 제거합니다.\n\n반환값:\n    value       - 범위 내이면 1.0(PASS), 그렇지 않으면 0.0(FAIL)\n    label       - "PASS" 또는 "FAIL"\n    explanation - 실제 길이와 허용 범위\n"""\n\nimport re\n\nfrom bedrock_agentcore.evaluation import (  # pylint: disable=no-name-in-module\n    EvaluatorInput,\n    EvaluatorOutput,\n    custom_code_based_evaluator,\n)\n\nMIN_LENGTH = 50\nMAX_LENGTH = 600\n\n_THINKING_RE = re.compile(r"<thinking>.*?</thinking>", re.DOTALL)\n\n\ndef _first_clean_message(span: dict) -> str:\n    """span 이벤트에서 정리된 첫 번째 비어 있지 않은 메시지를 반환합니다."""\n    for se in span.get("span_events", []):\n        body = se.get("body", {})\n        if not isinstance(body, dict):\n            continue\n        for msg in body.get("output", {}).get("messages", []):\n            content = msg.get("content", {})\n            if isinstance(content, dict):\n                text = content.get("message", "")\n                if text:\n                    cleaned = _THINKING_RE.sub("", text).strip()\n                    if cleaned:\n                        return cleaned\n    return ""\n\n\ndef _extract_final_response(spans: list) -> str:\n    """invoke_agent span에서 최종적으로 표시되는 응답 텍스트를 추출합니다."""\n    for span in spans:\n        name = (span.get("name") or "").lower()\n        if "invoke_agent" not in name:\n            continue\n        text = _first_clean_message(span)\n        if text:\n            return text\n    return ""\n\n\ndef _extract_fallback_response(spans: list) -> str:\n    """대체 방식: 모든 span_events에서 비어 있지 않은 content 메시지를 찾습니다."""\n    for span in reversed(spans):\n        for se in span.get("span_events", []):\n            body = se.get("body", {})\n            if not isinstance(body, dict):\n                continue\n            for msg in (body.get("output", {}) or {}).get("messages", []):\n                content = msg.get("content", {})\n                text = (\n                    (content.get("message") or "") if isinstance(content, dict) else ""\n                )\n                cleaned = _THINKING_RE.sub("", text).strip()\n                if cleaned and not cleaned.startswith("[{"):\n                    return cleaned\n    return ""\n\n\n@custom_code_based_evaluator()\ndef lambda_handler(evaluator_input: EvaluatorInput, _context) -> EvaluatorOutput:\n    """단일 에이전트 trace의 응답 길이를 평가합니다."""\n    spans = evaluator_input.session_spans\n\n    # TRACE 수준에서는 target_trace_id가 평가할 trace를 식별합니다.\n    if evaluator_input.evaluation_level == "TRACE" and evaluator_input.target_trace_id:\n        spans = [\n            s\n            for s in spans\n            if s.get("traceId") == evaluator_input.target_trace_id\n            or s.get("trace_id") == evaluator_input.target_trace_id\n        ]\n\n    output_text = _extract_final_response(spans) or _extract_fallback_response(spans)\n\n    if not output_text:\n        return EvaluatorOutput(\n            label="FAIL",\n            errorCode="NoResponseFound",\n            errorMessage=(\n                f"No agent response text found in {len(spans)} spans. "\n                "Expected invoke_agent span with span_events containing output message."\n            ),\n        )\n\n    length = len(output_text)\n\n    if MIN_LENGTH <= length <= MAX_LENGTH:\n        return EvaluatorOutput(\n            value=1.0,\n            label="PASS",\n            explanation=(\n                f"Response length {length} chars is within the acceptable range "\n                f"[{MIN_LENGTH}, {MAX_LENGTH}]."\n            ),\n        )\n    if length < MIN_LENGTH:\n        return EvaluatorOutput(\n            value=0.0,\n            label="FAIL",\n            explanation=(\n                f"Response length {length} chars is too short (minimum {MIN_LENGTH}). "\n                f\'Preview: "{output_text[:60]}..."\'\n            ),\n        )\n    return EvaluatorOutput(\n        value=0.0,\n        label="FAIL",\n        explanation=(\n            f"Response length {length} chars exceeds maximum {MAX_LENGTH}. "\n            "Consider a more concise answer."\n        ),\n    )\n'

with open("lambdas/hr_response_length/lambda_function.py", "w") as f:
    f.write(lambda_response_length_code)

print("Written: lambdas/hr_response_length/lambda_function.py")
print(f"  {len(lambda_response_length_code)} chars")

### Evaluator 2: HRFactChecker(SESSION 수준)

HR Assistant의 응답에 mock data store의 정확한 사실이 포함되어 있는지 결정론적으로 검증합니다.
정확한 패턴 일치를 사용하므로 LLM 추론이 없고 평가 자체에서 환각이 발생할
위험도 없습니다.

**SESSION 수준**에서 작동하므로 전체 대화의 모든 span을 받아 모든 턴의 사실을 확인합니다.

**확인하는 사실:**
- PTO balances: EMP-001 (10 remaining), EMP-002 (3 remaining), EMP-042 (13 remaining)
- Pay stubs: EMP-001 Jan 2026 gross=$8,333.33 / net=$5,362.50
- PTO request IDs must match format `PTO-2026-NNN`
- Policy facts: 15-day PTO accrual, 2-day advance notice, 401k 4% match, etc.

In [ ]:
os.makedirs("lambdas/hr_fact_checker", exist_ok=True)

lambda_fact_checker_code = '# pylint: disable=duplicate-code\n"""\nHRFactChecker - Code-Based Evaluator(SESSION 수준)\n\nSDK 1.6의 @custom_code_based_evaluator() 데코레이터를 사용합니다.\n알려진 ground truth 데이터를 기준으로 HR Assistant 응답을 결정론적으로 검증합니다.\nLLM-as-judge와 달리 이 evaluator는 정확한 패턴 일치를 사용합니다.\n\n반환값:\n    value       - 통과한 적용 가능 검사의 비율(0.0-1.0)\n    label       - "PASS"(모두), "PARTIAL"(>=50%), "FAIL"(<50%), "SKIP"(실행된 검사 없음)\n    explanation - 통과하거나 실패한 검사\n"""\n\nimport re\n\nfrom bedrock_agentcore.evaluation import (  # pylint: disable=no-name-in-module\n    EvaluatorInput,\n    EvaluatorOutput,\n    custom_code_based_evaluator,\n)\n\n# Ground truth 레지스트리(에이전트 모의 데이터와 동일)\nPTO_BALANCES = {\n    "EMP-001": {"remaining": 10, "total": 15, "used": 5},\n    "EMP-002": {"remaining": 3, "total": 15, "used": 12},\n    "EMP-042": {"remaining": 13, "total": 20, "used": 7},\n}\n\nPAY_STUBS = {\n    ("EMP-001", "2026-01"): {"gross": "8,333.33", "net": "5,362.50"},\n    ("EMP-001", "2025-12"): {"gross": "8,333.33", "net": "5,362.50"},\n    ("EMP-042", "2026-01"): {"gross": "10,416.67", "net": "6,607.30"},\n}\n\nMONTH_NAMES = {\n    "01": ["january", "jan"],\n    "02": ["february", "feb"],\n    "03": ["march", "mar"],\n    "04": ["april", "apr"],\n    "05": ["may"],\n    "06": ["june", "jun"],\n    "07": ["july", "jul"],\n    "08": ["august", "aug"],\n    "09": ["september", "sep"],\n    "10": ["october", "oct"],\n    "11": ["november", "nov"],\n    "12": ["december", "dec"],\n}\n\nPOLICY_FACTS_BY_TOPIC = {\n    "pto": [\n        (\n            "PTO accrual 15 days",\n            [r"15\\s*days?(\\s*of\\s*PTO|\\s*per\\s*year)", r"PTO.{0,30}15\\s*days?"],\n        ),\n        ("PTO advance notice 2 days", [r"2\\s*business\\s*days?", r"2-business-day"]),\n    ],\n    "remote_work": [\n        (\n            "Remote work 3 days/week",\n            [r"3\\s*days?\\s*(per|a)\\s*week", r"up\\s*to\\s*3\\s*days?"],\n        ),\n        (\n            "Core hours 10am-3pm",\n            [r"10\\s*[Aa]\\.?[Mm]\\.?.*3\\s*[Pp]\\.?[Mm]", r"10am.*3pm"],\n        ),\n    ],\n    "parental_leave": [\n        ("Primary leave 16 weeks", [r"16\\s*weeks?", r"primary.*16\\s*weeks?"]),\n        ("Secondary leave 6 weeks", [r"6\\s*weeks?", r"secondary.*6\\s*weeks?"]),\n    ],\n    "401k": [\n        ("401k 4% match", [r"4%?\\s*(of\\s*salary)?.*match", r"matches?\\s*4%"]),\n        ("401k 3-year vesting", [r"3[-\\s]year\\s*vest", r"vests?\\s*over\\s*3"]),\n    ],\n    "health": [\n        (\n            "Health 90% coverage",\n            [r"90%?\\s*(of\\s*premiums?|coverage)", r"covers?\\s*90%"],\n        ),\n    ],\n}\n\n_THINKING_RE = re.compile(r"<thinking>.*?</thinking>", re.DOTALL)\n\n\ndef _collect_span_texts(span: dict) -> list:\n    """단일 invoke_agent span에서 비어 있지 않은 모든 응답 텍스트를 추출합니다."""\n    texts = []\n    for se in span.get("span_events", []):\n        body = se.get("body", {})\n        if not isinstance(body, dict):\n            continue\n        for msg in body.get("output", {}).get("messages", []):\n            content = msg.get("content", {})\n            if isinstance(content, dict):\n                text = content.get("message", "")\n                if text:\n                    cleaned = _THINKING_RE.sub("", text).strip()\n                    if cleaned:\n                        texts.append(cleaned)\n    return texts\n\n\ndef _parse_spans(spans: list) -> tuple:\n    """세션 span에서 연결된 응답 텍스트와 도구 이름을 추출합니다."""\n    response_parts = []\n    tool_names = []\n    for span in spans:\n        name = (span.get("name") or "").lower()\n        attrs = span.get("attributes", {})\n        op = attrs.get("gen_ai.operation.name", "")\n        # 모든 invoke_agent 응답 텍스트를 수집합니다(멀티턴 세션).\n        if "invoke_agent" in name:\n            response_parts.extend(_collect_span_texts(span))\n        # 도구 호출\n        if op == "execute_tool":\n            tool_name = attrs.get("gen_ai.tool.name", "")\n            if tool_name:\n                tool_names.append(tool_name)\n    # 세션 수준의 사실 검사를 위해 모든 턴을 연결합니다.\n    all_text = " ".join(response_parts)\n    return all_text, tool_names\n\n\n@custom_code_based_evaluator()\ndef lambda_handler(  # pylint: disable=too-many-branches,too-many-locals,too-many-statements\n    evaluator_input: EvaluatorInput, _context\n) -> EvaluatorOutput:\n    """세션의 모든 턴에 걸쳐 HR 정보의 정확성을 검증합니다."""\n    spans = evaluator_input.session_spans\n    all_text, tool_names = _parse_spans(spans)\n    all_text_lower = all_text.lower()\n\n    checks_run, checks_passed, checks_failed = [], [], []\n\n    # 1. PTO 잔여 일수 정확성\n    if "get_pto_balance" in tool_names:\n        for emp_id, facts in PTO_BALANCES.items():\n            if emp_id not in all_text:\n                continue\n            correct = str(facts["remaining"])\n            near_remaining = rf"\\b{re.escape(correct)}\\b.{{0,30}}remaining"\n            near_value = rf"remaining.{{0,30}}\\b{re.escape(correct)}\\b"\n            remaining_pattern = re.compile(\n                f"{near_remaining}|{near_value}",\n                re.IGNORECASE,\n            )\n            check = f"PTO balance {emp_id} = {correct} remaining"\n            checks_run.append(check)\n            correct_val = bool(re.search(rf"\\b{re.escape(correct)}\\b", all_text))\n            if bool(remaining_pattern.search(all_text)) or (\n                correct_val and "remaining" in all_text_lower\n            ):\n                checks_passed.append(check)\n            else:\n                checks_failed.append(f"{check}: value not found near \'remaining\'")\n\n    # 2. 급여 명세서 정확성\n    for (emp_id, period), figures in PAY_STUBS.items():\n        if emp_id not in all_text:\n            continue\n        year, month_num = period.split("-")\n        month_variants = MONTH_NAMES.get(month_num, [])\n        if not (year in all_text and any(m in all_text_lower for m in month_variants)):\n            continue\n        for field, expected_val in [\n            ("gross", figures["gross"]),\n            ("net", figures["net"]),\n        ]:\n            check = f"Pay stub {emp_id} {period} {field} = ${expected_val}"\n            checks_run.append(check)\n            amount_re = re.compile(\n                r"\\$?\\s*" + re.escape(expected_val).replace(",", ",?"), re.IGNORECASE\n            )\n            if amount_re.search(all_text):\n                checks_passed.append(check)\n            else:\n                checks_failed.append(f"{check}: ${expected_val} not found")\n\n    # 3. PTO 요청 ID 형식\n    if "submit_pto_request" in tool_names:\n        check = "PTO request ID format PTO-2026-NNN"\n        checks_run.append(check)\n        match = re.search(r"PTO-2026-\\d{3}", all_text)\n        if match:\n            checks_passed.append(f"{check}: found {match.group()}")\n        else:\n            checks_failed.append(f"{check}: no PTO-2026-NNN ID found")\n\n    # 4. 정책 사실 검사\n    if "lookup_hr_policy" in tool_names or "get_benefits_summary" in tool_names:\n        kw_topic_map = [\n            ("pto", ["paid time off", "pto policy", "pto accrual"]),\n            ("remote_work", ["remote work", "work remotely"]),\n            ("parental_leave", ["parental leave", "maternity", "paternity"]),\n            ("401k", ["401(k)", "401k", "employer match"]),\n            ("health", ["health insurance", "health plan", "hmo", "ppo", "hdhp"]),\n        ]\n        for topic, keywords in kw_topic_map:\n            if not any(kw in all_text_lower for kw in keywords):\n                continue\n            for check_desc, patterns in POLICY_FACTS_BY_TOPIC.get(topic, []):\n                check = f"Policy fact: {check_desc}"\n                checks_run.append(check)\n                if any(re.search(p, all_text, re.IGNORECASE) for p in patterns):\n                    checks_passed.append(check)\n                else:\n                    checks_failed.append(f"{check}: expected phrase not found")\n\n    if not checks_run:\n        return EvaluatorOutput(\n            value=1.0,\n            label="SKIP",\n            explanation=(\n                f"No applicable checks triggered. "\n                f"Tools: {tool_names or [\'none\']}, response length: {len(all_text)} chars."\n            ),\n        )\n\n    total = len(checks_run)\n    passed = len(checks_passed)\n    value = round(passed / total, 3)\n    label = "PASS" if value == 1.0 else ("PARTIAL" if value >= 0.5 else "FAIL")\n\n    lines = [f"{passed}/{total} HR fact checks passed."]\n    if checks_passed:\n        lines.append("Passed: " + "; ".join(checks_passed[:4]))\n    if checks_failed:\n        lines.append("Failed: " + "; ".join(checks_failed[:4]))\n\n    return EvaluatorOutput(value=value, label=label, explanation=" | ".join(lines))\n'

with open("lambdas/hr_fact_checker/lambda_function.py", "w") as f:
    f.write(lambda_fact_checker_code)

print("Written: lambdas/hr_fact_checker/lambda_function.py")
print(f"  {len(lambda_fact_checker_code)} chars")

## 5단계: Lambda 함수 배포

각 Lambda 함수를 zip 아카이브로 패키징해 AWS Lambda에 배포합니다.
그런 다음 AgentCore Evaluations에 함수 호출 권한을 부여하는 리소스 정책을 추가합니다.

In [ ]:
lambda_client = boto3.client("lambda", region_name=EVAL_REGION)


def make_zip(source_dir: str) -> bytes:
    """Lambda 소스, bedrock-agentcore SDK, pydantic을 메모리 내 zip으로 패키징합니다.

    Lambda 함수는 bedrock_agentcore와 pydantic이 필요한 SDK 1.6의
    @custom_code_based_evaluator() 데코레이터를 사용합니다. 필요한 패키지를 모두 묶습니다.
    boto3/botocore는 Lambda 런타임에 이미 있으므로 공간 절약을 위해 제외합니다.
    """
    buf = io.BytesIO()
    with tempfile.TemporaryDirectory() as tmpdir:
        pkg_dir = Path(tmpdir) / "packages"
        pkg_dir.mkdir()

        print("  Bundling bedrock-agentcore SDK (no runtime deps — boto3 is in Lambda runtime)...")
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "bedrock-agentcore>=1.6.0",
                "--no-deps",
                "--target",
                str(pkg_dir),
                "--quiet",
            ],
            check=True,
        )

        print("  Bundling pydantic (Linux x86_64 binary for Python 3.12)...")
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "pydantic>=2.0.0",
                "--target",
                str(pkg_dir),
                "--platform",
                "manylinux2014_x86_64",
                "--implementation",
                "cp",
                "--python-version",
                "312",
                "--only-binary=:all:",
                "--quiet",
            ],
            check=True,
        )

        # evaluator decorator에 꼭 필요하지는 않지만 bedrock-agentcore가 starlette,
        # uvicorn, websockets, typing-extensions를 가져옵니다.
        print("  Bundling starlette, uvicorn, websockets, typing-extensions...")
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "starlette",
                "uvicorn",
                "websockets",
                "typing-extensions",
                "--target",
                str(pkg_dir),
                "--quiet",
            ],
            check=True,
        )

        with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
            # Lambda 함수 소스 파일
            for py_file in sorted(Path(source_dir).glob("*.py")):
                zf.write(py_file, py_file.name)
                print(f"    + {py_file.name} ({py_file.stat().st_size} bytes)")
            # 번들 패키지
            for pkg_file in sorted(pkg_dir.rglob("*")):
                if pkg_file.is_file():
                    zf.write(pkg_file, str(pkg_file.relative_to(pkg_dir)))

    buf.seek(0)
    data = buf.read()
    print(f"  Total zip size: {len(data):,} bytes ({len(data) // 1024} KB)")
    return data


def deploy_lambda(function_name: str, source_dir: str, timeout_s: int = 60) -> str:
    """Lambda 함수를 생성하거나 업데이트하고 함수 ARN을 반환합니다."""
    print(f"\n[Lambda] Packaging {function_name}...")
    zip_bytes = make_zip(source_dir)

    try:
        resp = lambda_client.get_function(FunctionName=function_name)
        print("  Function exists — updating code...")
        lambda_client.update_function_code(FunctionName=function_name, ZipFile=zip_bytes)
        waiter = lambda_client.get_waiter("function_updated_v2")
        waiter.wait(FunctionName=function_name)
        arn = resp["Configuration"]["FunctionArn"]
        print(f"  Updated: {arn}")
    except lambda_client.exceptions.ResourceNotFoundException:
        print("  Creating new Lambda function...")
        resp = lambda_client.create_function(
            FunctionName=function_name,
            Runtime="python3.12",
            Role=LAMBDA_ROLE_ARN,
            Handler="lambda_function.lambda_handler",
            Code={"ZipFile": zip_bytes},
            Timeout=timeout_s + 10,
            MemorySize=128,
            Description=f"AgentCore code-based evaluator: {function_name}",
        )
        waiter = lambda_client.get_waiter("function_active_v2")
        waiter.wait(FunctionName=function_name)
        arn = resp["FunctionArn"]
        print(f"  Created: {arn}")
    return arn


def add_invoke_permission(function_name: str) -> None:
    """AgentCore Evaluate 서비스 보안 주체에 Lambda 호출 권한을 부여합니다."""
    statement_id = "AllowAgentCoreEvaluateInvoke"
    print(f"  Adding resource policy to {function_name}...")
    try:
        lambda_client.remove_permission(FunctionName=function_name, StatementId=statement_id)
    except lambda_client.exceptions.ResourceNotFoundException:
        pass
    lambda_client.add_permission(
        FunctionName=function_name,
        StatementId=statement_id,
        Action="lambda:InvokeFunction",
        Principal="bedrock-agentcore.amazonaws.com",
        SourceAccount=ACCOUNT_ID,
    )
    print("  Granted lambda:InvokeFunction to bedrock-agentcore.amazonaws.com")

In [ ]:
# 두 Lambda 함수 배포
lambda_arn_response_length = deploy_lambda(
    "hr-response-length",
    "lambdas/hr_response_length",
    timeout_s=30,
)
add_invoke_permission("hr-response-length")

lambda_arn_fact_checker = deploy_lambda(
    "hr-fact-checker",
    "lambdas/hr_fact_checker",
    timeout_s=60,
)
add_invoke_permission("hr-fact-checker")

print("\nLambda functions deployed:")
print(f"  hr-response-length : {lambda_arn_response_length}")
print(f"  hr-fact-checker    : {lambda_arn_fact_checker}")

## 6단계: AgentCore에 evaluator 등록

Lambda 함수를 배포한 후 AgentCore Evaluations control plane API를 사용해 evaluator로
등록합니다. 그러면 Lambda ARN을 참조하는 evaluator 레코드가 생성됩니다.

`level` 필드는 evaluator 호출 방식을 제어합니다.
- **TRACE** - 에이전트 응답(trace)마다 한 번 호출
- **SESSION** - 대화 세션마다 한 번 호출


In [ ]:
# SDK 1.6: bedrock-agentcore-control은 표준 boto3 서비스이므로 사용자 지정 엔드포인트가 필요하지 않습니다.
cp_client = boto3.client(
    "bedrock-agentcore-control",
    region_name=EVAL_REGION,
    config=Config(retries={"max_attempts": 3, "mode": "standard"}),
)

In [ ]:
def create_code_evaluator(name: str, lambda_arn: str, level: str, timeout_s: int) -> dict:
    """Lambda 함수를 AgentCore 코드 기반 evaluator로 등록합니다.

    재실행 시 ConflictException을 피하도록 이름에 고유한 실행 접미사를 붙입니다.
    """
    unique_name = f"{name}_{RUN_SUFFIX}"
    print(f"[Evaluator] Creating '{unique_name}' (level={level})...")
    resp = cp_client.create_evaluator(
        evaluatorName=unique_name,
        level=level,
        evaluatorConfig={
            "codeBased": {
                "lambdaConfig": {
                    "lambdaArn": lambda_arn,
                    "lambdaTimeoutInSeconds": timeout_s,
                }
            }
        },
    )
    evaluator_id = resp["evaluatorId"]
    print(f"  evaluatorId : {evaluator_id}")
    print(f"  level       : {level}")
    return {"id": evaluator_id, "level": level, "lambda_arn": lambda_arn}

In [ ]:
# 다시 실행할 때 기존 evaluator 이름과 충돌하지 않도록 짧은 고유 접미사 생성
RUN_SUFFIX = uuid.uuid4().hex[:8]
print(f"Run suffix: {RUN_SUFFIX}")

# IAM 정책 전파를 위해 잠시 대기
time.sleep(5)

response_length_eval = create_code_evaluator(
    name="HRResponseLength",
    lambda_arn=lambda_arn_response_length,
    level="TRACE",
    timeout_s=30,
)

fact_checker_eval = create_code_evaluator(
    name="HRFactChecker",
    lambda_arn=lambda_arn_fact_checker,
    level="SESSION",
    timeout_s=60,
)

CODE_EVAL_IDS = {
    "HRResponseLength": response_length_eval["id"],
    "HRFactChecker": fact_checker_eval["id"],
}

print("\nCode-based evaluators registered:")
for name, eid in CODE_EVAL_IDS.items():
    print(f"  {name} : {eid}")

In [ ]:
# 나중에 사용할 evaluator ID 저장
os.makedirs("results", exist_ok=True)
ids_path = "results/code_evaluator_ids.json"
with open(ids_path, "w") as f:
    json.dump(
        {
            "HRResponseLength": {
                "id": response_length_eval["id"],
                "level": "TRACE",
                "lambda_arn": lambda_arn_response_length,
            },
            "HRFactChecker": {
                "id": fact_checker_eval["id"],
                "level": "SESSION",
                "lambda_arn": lambda_arn_fact_checker,
            },
        },
        f,
        indent=2,
    )
print(f"Evaluator IDs saved to: {ids_path}")

## 7단계: 코드 기반 evaluator를 사용한 온디맨드 평가

온디맨드 평가를 사용하면 에이전트를 다시 호출하지 않고도 CloudWatch의 특정 기존 세션을 평가할 수 있습니다.
`session_id`와 evaluator ID 목록을 제공하면 서비스가 저장된 OTel span을 가져와 evaluator를 실행합니다.

먼저 HR Assistant를 호출해 새 세션을 생성한 다음, 비교를 위해 두 개의 기본 제공 evaluator와 함께 **코드 기반 evaluator ID**(`HRResponseLength`, `HRFactChecker`)를 `EvaluationClient.run()`에 전달합니다.

| Evaluator | 유형 | 수준 | 확인 항목 |
|---|---|---|---|
| `Builtin.Correctness` | LLM | TRACE | 예상 답변과의 의미적 유사성 |
| `Builtin.GoalSuccessRate` | LLM | SESSION | 에이전트의 사용자 목표 달성 여부 |
| `HRResponseLength_<suffix>` | 코드 | TRACE | 응답 길이 50~600자 |
| `HRFactChecker_<suffix>` | 코드 | SESSION | PTO 잔여 일수, 급여 명세서, 정책의 정확성 |

> **참고:** 에이전트가 `EVAL_REGION`과 다른 리전에 있으면 `EvaluationClient`가 CloudWatch span을
> 수집할 때 에이전트 리전이 필요합니다. 해결 방법은 `EVAL_REGION`으로 클라이언트를 생성해 data plane이
> `EVAL_REGION`의 evaluator를 대상으로 하게 한 다음, CloudWatch 쿼리가 에이전트 리전에서 실행되도록
> `ec.region_name = REGION`을 설정하는 것입니다.


In [ ]:
# 온디맨드 세션용 간단한 invoker(AgentInvokerInput wrapper 불필요)
def invoke_agent_simple(prompt: str, session_id: str) -> str:
    resp = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ARN,
        qualifier="DEFAULT",
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": prompt}).encode("utf-8"),
    )
    raw = resp["response"].read().decode("utf-8")
    parts = []
    for line in raw.splitlines():
        if line.startswith("data: "):
            chunk = line[len("data: ") :]
            try:
                chunk = json.loads(chunk)
            except Exception:
                pass
            parts.append(str(chunk))
    return "".join(parts) if parts else raw

In [ ]:
# 알려진 HR 데이터 사실이 포함된 세션을 생성하도록 에이전트 호출
ONDEMAND_SESSION_ID = f"ondemand-eval-{uuid.uuid4()}"
print(f"Session ID : {ONDEMAND_SESSION_ID}")

turns = [
    "What is the current PTO balance for employee EMP-001?",
    "Please submit a PTO request for EMP-001 from 2026-06-02 to 2026-06-04.",
    "What is the company PTO policy?",
]

for prompt in turns:
    print(f"  > {prompt}")
    reply = invoke_agent_simple(prompt, ONDEMAND_SESSION_ID)
    print(f"  < {reply[:120]}")

print("\nWaiting 90s for CloudWatch log ingestion...")
time.sleep(90)
print("Ready for on-demand evaluation.")

In [ ]:
from bedrock_agentcore.evaluation import EvaluationClient
from datetime import timedelta

# data plane 호출이 코드 기반 evaluator가 등록된 리전으로 전달되도록
# EVAL_REGION으로 EvaluationClient를 생성합니다.
ec = EvaluationClient(region_name=EVAL_REGION)

# 에이전트가 다른 리전에 있으면 내부 CloudWatchAgentSpanCollector가 올바른 리전에서
# span을 쿼리하도록 region_name을 재정의합니다.
if REGION != EVAL_REGION:
    ec.region_name = REGION  # CW span 조회 → 에이전트 리전

# 추가 GetEvaluator API 호출 없이 SDK가 evaluator 수준을 확인할 수 있도록
# 수준 캐시를 미리 채웁니다(Builtin.* evaluator에 필요).
ec._evaluator_level_cache.update(
    {
        "Builtin.Correctness": "TRACE",
        "Builtin.GoalSuccessRate": "SESSION",
        CODE_EVAL_IDS["HRResponseLength"]: "TRACE",
        CODE_EVAL_IDS["HRFactChecker"]: "SESSION",
    }
)

ondemand_evaluators = [
    "Builtin.Correctness",
    "Builtin.GoalSuccessRate",
    CODE_EVAL_IDS["HRResponseLength"],  # Lambda: TRACE 수준
    CODE_EVAL_IDS["HRFactChecker"],  # Lambda: SESSION 수준
]

print(f"Running on-demand evaluation for session: {ONDEMAND_SESSION_ID}")
print(f"  CW region   : {ec.region_name} (agent spans)")
print(f"  Eval region : {EVAL_REGION} (code-based evaluators)")
print(f"  Evaluators  : {ondemand_evaluators}\n")

ondemand_results = ec.run(
    evaluator_ids=ondemand_evaluators,
    session_id=ONDEMAND_SESSION_ID,
    agent_id=AGENT_ID,
    look_back_time=timedelta(hours=2),
)

print(f"\nEvaluation complete. {len(ondemand_results)} result(s) returned.")

In [ ]:
# EvaluationClient.run()은 Ground Truth Notebook과 동일한 스키마의 List[dict]를 반환합니다
name_by_id = {v: k for k, v in CODE_EVAL_IDS.items()}

rows = ["| Evaluator | Type | Value | Label | Explanation |", "|---|---|---|---|---|"]

for res in ondemand_results:
    evaluator_id = res.get("evaluatorId", "")
    kind = "code" if evaluator_id in CODE_EVAL_IDS.values() else "builtin"
    display_name = name_by_id.get(evaluator_id, evaluator_id)
    value = str(res.get("value", res.get("score", "N/A")))
    lbl = str(res.get("label", res.get("rating", "")))
    explanation = (res.get("explanation", "") or "")[:120].replace("\n", " ")
    error_code = res.get("errorCode")
    if error_code:
        lbl = f"ERR:{error_code}"
        explanation = (res.get("errorMessage", "") or "")[:120]
    rows.append(f"| `{display_name}` | {kind} | {value} | {lbl} | {explanation} |")

display(Markdown("### On-Demand Evaluation Results\n\n" + "\n".join(rows)))

## 8단계: 혼합 evaluator 세트로 평가 실행

이제 방금 등록한 두 개의 코드 기반 Lambda evaluator와 세 개의 기본 제공 LLM evaluator를 조합한 **혼합 evaluator 세트**로 `OnDemandEvaluationDatasetRunner`를 실행합니다.

`OnDemandEvaluationDatasetRunner`는 모든 작업을 자동으로 처리합니다.
1. 데이터 세트의 각 시나리오 순회
2. 시나리오 prompt로 에이전트 호출
3. CloudWatch 로그가 도착할 때까지 대기
4. 결과 span에 각 evaluator 호출
5. 점수를 `EvaluationResult`로 집계

```
데이터 세트 → OnDemandEvaluationDatasetRunner → 에이전트 호출 → CloudWatch → Evaluator → 결과
```

In [ ]:
from bedrock_agentcore.evaluation import (
    AgentInvokerInput,
    AgentInvokerOutput,
    CloudWatchAgentSpanCollector,
    Dataset,
    EvaluationRunConfig,
    OnDemandEvaluationDatasetRunner,
    EvaluatorConfig,
    Turn,
    PredefinedScenario,
)

In [ ]:
def agent_invoker(invoker_input: AgentInvokerInput) -> AgentInvokerOutput:
    """HR Assistant를 호출하고 응답 텍스트를 반환합니다."""
    payload = invoker_input.payload
    body = {"prompt": payload} if isinstance(payload, str) else payload

    resp = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ARN,
        qualifier="DEFAULT",
        runtimeSessionId=invoker_input.session_id,
        payload=json.dumps(body).encode("utf-8"),
    )

    raw = resp["response"].read().decode("utf-8")
    parts = []
    for line in raw.splitlines():
        if line.startswith("data: "):
            chunk = line[len("data: ") :]
            try:
                chunk = json.loads(chunk)
            except Exception:
                pass
            parts.append(str(chunk))
    return AgentInvokerOutput(agent_output="".join(parts) if parts else raw)

In [ ]:
# 평가 데이터 세트 - HRFactChecker가 검증하는 사실을 테스트하는 시나리오
dataset = Dataset(
    scenarios=[
        PredefinedScenario(
            scenario_id="pto-balance-check",
            turns=[
                Turn(
                    input="What is the current PTO balance for employee EMP-001?",
                    expected_response="Employee EMP-001 has 10 remaining PTO days out of 15 total (5 days used).",
                )
            ],
            expected_trajectory=["get_pto_balance"],
            assertions=[
                "Agent called get_pto_balance with employee_id=EMP-001",
                "Agent reported 10 remaining PTO days",
            ],
        ),
        PredefinedScenario(
            scenario_id="submit-pto-request",
            turns=[
                Turn(
                    input="Please submit a PTO request for EMP-001 from 2026-04-14 to 2026-04-16 for a family vacation.",
                    expected_response="PTO request submitted and approved for EMP-001 from 2026-04-14 to 2026-04-16.",
                )
            ],
            expected_trajectory=["submit_pto_request"],
            assertions=[
                "Agent called submit_pto_request for EMP-001",
                "Agent confirmed the request was approved and provided a request ID",
            ],
        ),
        PredefinedScenario(
            scenario_id="pay-stub-lookup",
            turns=[
                Turn(
                    input="Can you pull up the January 2026 pay stub for employee EMP-001?",
                    expected_response="EMP-001 January 2026: gross pay $8,333.33, net pay $5,362.50.",
                )
            ],
            expected_trajectory=["get_pay_stub"],
            assertions=[
                "Agent called get_pay_stub for EMP-001 period 2026-01",
                "Agent reported gross pay of $8,333.33",
                "Agent reported net pay of $5,362.50",
            ],
        ),
        PredefinedScenario(
            scenario_id="pto-policy-lookup",
            turns=[
                Turn(
                    input="What is the company PTO policy?",
                    expected_response="Full-time employees accrue 15 days of PTO per year. Requests must be submitted at least 2 business days in advance.",
                )
            ],
            expected_trajectory=["lookup_hr_policy"],
            assertions=[
                "Agent called lookup_hr_policy with topic=pto",
                "Agent mentioned 15 days annual accrual",
                "Agent mentioned the 2 business day advance notice requirement",
            ],
        ),
        PredefinedScenario(
            scenario_id="health-benefits",
            turns=[
                Turn(
                    input="Can you tell me about the company health insurance options?",
                    expected_response="The company covers 90% of health insurance premiums for employee-only coverage. Three plans are available: Blue Shield PPO, Kaiser HMO, and HDHP with HSA.",
                )
            ],
            expected_trajectory=["get_benefits_summary"],
            assertions=[
                "Agent called get_benefits_summary with benefit_type=health",
                "Agent mentioned 90% premium coverage",
            ],
        ),
    ]
)

print(f"Dataset: {len(dataset.scenarios)} scenarios")

In [ ]:
# 혼합 evaluator 세트: 기본 제공 LLM evaluator + 코드 기반 Lambda evaluator
builtin_evaluator_ids = [
    "Builtin.Correctness",  # TRACE - expected_response 사용
    "Builtin.Helpfulness",  # TRACE - Ground Truth 불필요
    "Builtin.ResponseRelevance",  # TRACE - Ground Truth 불필요
]

code_evaluator_ids = list(CODE_EVAL_IDS.values())

all_evaluator_ids = builtin_evaluator_ids + code_evaluator_ids

# 추가 API 호출 없이 runner가 각 evaluator의 TRACE 또는 SESSION 실행 수준을
# 알 수 있도록 evaluator 수준 캐시를 미리 채웁니다.
# Builtin.* evaluator는 GetEvaluator로 확인할 수 없으므로 필요합니다.
EVALUATOR_LEVELS = {
    "Builtin.Correctness": "TRACE",
    "Builtin.Helpfulness": "TRACE",
    "Builtin.ResponseRelevance": "TRACE",
    "Builtin.GoalSuccessRate": "SESSION",
    "Builtin.TrajectoryExactOrderMatch": "SESSION",
    CODE_EVAL_IDS["HRResponseLength"]: "TRACE",  # 코드 기반 TRACE
    CODE_EVAL_IDS["HRFactChecker"]: "SESSION",  # 코드 기반 SESSION
}

span_collector = CloudWatchAgentSpanCollector(
    log_group_name=CW_LOG_GROUP,
    region=REGION,
    max_wait_seconds=180,
    poll_interval_seconds=15,
)

config = EvaluationRunConfig(
    evaluator_config=EvaluatorConfig(evaluator_ids=all_evaluator_ids),
    evaluation_delay_seconds=180,
    max_concurrent_scenarios=3,
)

runner = OnDemandEvaluationDatasetRunner(region=EVAL_REGION)
runner._evaluator_level_cache.update(EVALUATOR_LEVELS)

print(f"Evaluator set ({len(all_evaluator_ids)} total):")
for eid in all_evaluator_ids:
    level = EVALUATOR_LEVELS.get(eid, "?")
    kind = "code-based" if eid in code_evaluator_ids else "builtin"
    print(f"  [{level}] {eid} ({kind})")

In [ ]:
print("Starting evaluation...")
print(f"  Scenarios : {len(dataset.scenarios)}")
print(
    f"  Evaluators: {len(all_evaluator_ids)} ({len(builtin_evaluator_ids)} builtin + {len(code_evaluator_ids)} code-based)"
)
print(f"  Delay     : {config.evaluation_delay_seconds}s\n")

eval_result = runner.run(
    config=config,
    dataset=dataset,
    agent_invoker=agent_invoker,
    span_collector=span_collector,
)

completed = sum(1 for sr in eval_result.scenario_results if sr.status == "COMPLETED")
failed = sum(1 for sr in eval_result.scenario_results if sr.status == "FAILED")
print(f"\nEvaluation complete: {completed} completed, {failed} failed.")

## 9단계: 결과 확인

결과를 두 가지 보기로 표시합니다.
1. **전체 표** - 모든 evaluator와 시나리오 및 설명
2. **코드 evaluator 요약** - 두 Lambda 기반 evaluator의 집계 점수

In [ ]:
def display_all_results(eval_result) -> None:
    """결과를 시나리오별 Markdown 표로 표시합니다."""
    for sr in eval_result.scenario_results:
        if sr.status == "FAILED":
            display(Markdown(f"**Scenario `{sr.scenario_id}`** — FAILED: {sr.error}"))
            continue

        rows = [
            "| Evaluator | Type | Value | Label | Explanation |",
            "|---|---|---|---|---|",
        ]
        for er in sr.evaluator_results:
            kind = "code" if er.evaluator_id in code_evaluator_ids else "builtin"
            for res in er.results:
                value = str(res.get("value", res.get("score", "N/A")))
                lbl = str(res.get("label", res.get("rating", "")))
                explanation = (res.get("explanation", "") or "")[:110].replace("\n", " ")
                error_code = res.get("errorCode")
                if error_code:
                    lbl = f"ERR:{error_code}"
                    explanation = (res.get("errorMessage", "") or "")[:110]
                rows.append(f"| `{er.evaluator_id[:38]}` | {kind} | {value} | {lbl} | {explanation} |")

        display(Markdown(f"### Scenario: `{sr.scenario_id}`\n\n" + "\n".join(rows)))


display_all_results(eval_result)

In [ ]:
def display_code_eval_summary(eval_result, code_eval_ids: dict) -> None:
    """모든 시나리오의 코드 기반 evaluator 결과를 간추려 보여 줍니다."""
    code_eval_set = set(code_eval_ids.values())
    name_by_id = {v: k for k, v in code_eval_ids.items()}

    rows = [
        "| Scenario | Evaluator | Value | Label | Explanation |",
        "|---|---|---|---|---|",
    ]

    for sr in eval_result.scenario_results:
        if sr.status != "COMPLETED":
            continue
        for er in sr.evaluator_results:
            if er.evaluator_id not in code_eval_set:
                continue
            eval_name = name_by_id.get(er.evaluator_id, er.evaluator_id)
            for res in er.results:
                value = str(res.get("value", "N/A"))
                lbl = str(res.get("label", ""))
                explanation = (res.get("explanation", "") or "")[:120].replace("\n", " ")
                error_code = res.get("errorCode")
                if error_code:
                    lbl = f"ERR:{error_code}"
                    explanation = (res.get("errorMessage", "") or "")[:120]
                rows.append(f"| `{sr.scenario_id}` | **{eval_name}** | {value} | {lbl} | {explanation} |")

    display(Markdown("## Code-Based Evaluator Summary\n\n" + "\n".join(rows)))


display_code_eval_summary(eval_result, CODE_EVAL_IDS)

In [ ]:
# 코드 기반 evaluator와 기본 제공 evaluator의 평균 점수 비교
from collections import defaultdict

builtin_scores = defaultdict(list)
code_scores = defaultdict(list)

for sr in eval_result.scenario_results:
    if sr.status != "COMPLETED":
        continue
    for er in sr.evaluator_results:
        for res in er.results:
            if "value" in res and res["value"] is not None and not res.get("errorCode"):
                score = float(res["value"])
                if er.evaluator_id in code_evaluator_ids:
                    code_scores[er.evaluator_id].append(score)
                else:
                    builtin_scores[er.evaluator_id].append(score)

print("Built-in (LLM-as-judge) evaluator averages:")
for eid, scores in sorted(builtin_scores.items()):
    print(f"  {eid:<45} avg={sum(scores) / len(scores):.2f}  (n={len(scores)})")

print("\nCode-based (Lambda) evaluator averages:")
name_by_id = {v: k for k, v in CODE_EVAL_IDS.items()}
for eid, scores in sorted(code_scores.items()):
    name = name_by_id.get(eid, eid)
    print(f"  {name:<45} avg={sum(scores) / len(scores):.2f}  (n={len(scores)})")

In [ ]:
timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
results_path = f"results/code_based_eval_{timestamp}.json"
with open(results_path, "w") as f:
    json.dump(eval_result.model_dump(), f, indent=2, default=str)
print(f"Results saved to: {results_path}")

## 10단계: 코드 기반 evaluator를 사용한 온라인 평가

**온라인 평가**는 실시간 에이전트 트래픽을 지속적으로 모니터링하고 세션이 발생할 때마다
자동으로 점수를 산출하므로 수동 트리거가 필요하지 않습니다. 한 번 구성하면 AgentCore가 에이전트의
CloudWatch 로그 스트림을 모니터링하고 구성 가능한 샘플링 비율로 새 세션을 평가합니다.

이 단계에서는 6단계에서 등록한 **동일한 코드 기반 evaluator**(`HRResponseLength`, `HRFactChecker`)를
재사용합니다. 이를 통해 하나의 evaluator 등록으로 온디맨드 및 온라인 평가 사용 사례를 모두
지원할 수 있음을 보여 줍니다.

### 온라인 평가 작동 방식

```
에이전트 호출
      │
      ▼  (OTel span → CloudWatch)
AgentCore Runtime 로그 그룹
      │
      ▼  (온라인 평가 구성이 로그 그룹 모니터링)
AgentCore Evaluations
      ├── 기본 제공 LLM evaluator  → LLM 추론
      └── 코드 기반 evaluator      → 사용자 Lambda 함수
             │
             ▼
       CloudWatch Logs의 결과
       /aws/bedrock-agentcore/evaluations/online-evaluations/...
```

### 온디맨드 평가와의 주요 차이점

| | 온디맨드 | 온라인 |
|---|---|---|
| **트리거** | 세션별 명시적 API 호출 | 자동, 이벤트 기반 |
| **범위** | 선택한 특정 세션 | 모든 세션 또는 샘플링된 비율 |
| **설정** | 세션마다 `EvaluationClient.run()` 호출 | `create_online_evaluation_config`로 한 번 구성 |
| **Evaluator 잠금** | 없음 | 구성이 활성화된 동안 코드 기반 evaluator가 **잠김** |
| **적합한 용도** | 임시 검사, CI/CD 파이프라인 | 지속적인 프로덕션 모니터링 |

### IAM 실행 역할

온라인 평가에는 AgentCore Evaluations가 Lambda evaluator를 호출하고 CloudWatch span을 읽기 위해
수임하는 IAM 역할인 **평가 실행 역할**이 필요합니다. 이 역할은 `bedrock-agentcore.amazonaws.com`을
신뢰해야 하며 `lambda:InvokeFunction` + `logs:FilterLogEvents` 권한이 있어야 합니다.

> **Evaluator 잠금:** 활성화된 온라인 평가 구성에서 코드 기반 evaluator를 참조하면 실수로 수정되지
> 않도록 AgentCore가 자동으로 잠급니다. evaluator를 업데이트하려면 먼저 온라인 평가 구성을 비활성화하거나
> 삭제한 다음 evaluator를 업데이트하고 구성을 다시 활성화하세요.

### 10a단계: IAM 평가 실행 역할 생성

In [ ]:
iam_client = boto3.client("iam")
ONLINE_EVAL_ROLE_NAME = "AgentCoreOnlineEvaluationRole"
ONLINE_EVAL_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{ONLINE_EVAL_ROLE_NAME}"

# 신뢰 정책: AgentCore Evaluations 서비스가 이 역할을 수임하도록 허용
trust_policy = json.dumps(
    {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
                "Action": "sts:AssumeRole",
            }
        ],
    }
)

# 인라인 권한 정책: Lambda evaluator 호출 + 전체 CloudWatch Logs 액세스입니다.
# 온라인 평가 서비스에는 다음 권한이 필요합니다.
#   읽기 - 다음 대상에 대한 FilterLogEvents, GetLogEvents, StartQuery, GetQueryResults:
#             - agent runtime 로그 그룹(/aws/bedrock-agentcore/runtimes/...)
#             - OTel span 로그 그룹(aws/spans - 앞에 슬래시 없음)
#   쓰기 - 평가 결과 기록을 위한 CreateLogGroup, CreateLogStream, PutLogEvents:
#             - /aws/bedrock-agentcore/evaluations/results/<config-name>
eval_policy = json.dumps(
    {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "InvokeLambdaEvaluators",
                "Effect": "Allow",
                "Action": ["lambda:InvokeFunction", "lambda:GetFunction"],
                "Resource": [
                    lambda_arn_response_length,
                    lambda_arn_fact_checker,
                ],
            },
            {
                "Sid": "CloudWatchLogsReadSpans",
                "Effect": "Allow",
                "Action": [
                    "logs:FilterLogEvents",
                    "logs:DescribeLogGroups",
                    "logs:DescribeLogStreams",
                    "logs:GetLogEvents",
                    "logs:StartQuery",
                    "logs:GetQueryResults",
                    "logs:StopQuery",
                ],
                "Resource": "*",
            },
            {
                "Sid": "CloudWatchLogsWriteResults",
                "Effect": "Allow",
                "Action": [
                    "logs:CreateLogGroup",
                    "logs:CreateLogStream",
                    "logs:PutLogEvents",
                ],
                "Resource": f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:/aws/bedrock-agentcore/evaluations/*",
            },
        ],
    }
)

try:
    iam_client.get_role(RoleName=ONLINE_EVAL_ROLE_NAME)
    print(f"Using existing role: {ONLINE_EVAL_ROLE_ARN}")
    iam_client.put_role_policy(
        RoleName=ONLINE_EVAL_ROLE_NAME,
        PolicyName="AgentCoreOnlineEvalPermissions",
        PolicyDocument=eval_policy,
    )
    print("  Inline policy updated.")
except iam_client.exceptions.NoSuchEntityException:
    print(f"Creating IAM role: {ONLINE_EVAL_ROLE_NAME}...")
    iam_client.create_role(
        RoleName=ONLINE_EVAL_ROLE_NAME,
        AssumeRolePolicyDocument=trust_policy,
        Description="Execution role for AgentCore online evaluation with code-based evaluators",
    )
    iam_client.put_role_policy(
        RoleName=ONLINE_EVAL_ROLE_NAME,
        PolicyName="AgentCoreOnlineEvalPermissions",
        PolicyDocument=eval_policy,
    )
    print(f"Created: {ONLINE_EVAL_ROLE_ARN}")

print(f"\nOnline eval execution role: {ONLINE_EVAL_ROLE_ARN}")
print("Waiting 10s for IAM propagation...")
time.sleep(10)

### 10b단계: 온라인 평가 구성 생성

HR Assistant의 실시간 CloudWatch 로그 그룹을 모니터링하고 두 코드 기반 evaluator로 모든 세션을
평가하는 온라인 평가 구성(샘플링 비율 100%)을 생성합니다.

이 구성은 다음을 참조합니다.
- **`HRResponseLength`**(TRACE 수준) - 에이전트 응답 턴마다 평가
- **`HRFactChecker`**(SESSION 수준) - 완료된 세션마다 한 번 평가

> 이 구성을 **활성화**하면 두 코드 기반 evaluator가 자동으로 **잠기며**, 구성을 비활성화하거나
> 삭제할 때까지 수정할 수 없습니다.

In [ ]:
# 온라인 평가 구성의 고유 이름(하이픈 사용 불가, 서비스 정규식: [a-zA-Z][a-zA-Z0-9_]{0,99})
ONLINE_EVAL_CONFIG_NAME = f"hr_online_eval_{RUN_SUFFIX}"

# OTel 서비스 이름은 <agentRuntimeName>.DEFAULT입니다
# AGENT_ARN 형식: arn:aws:bedrock-agentcore:{region}:{account}:runtime/{id}
_runtime_id = AGENT_ARN.split("/")[-1]  # 예: hr_assistant_codeeval_tutorial-AbCdEfGhIj
_agent_runtime_name = _runtime_id.rsplit("-", 1)[0]  # 자동 생성된 접미사 제거
OTEL_SERVICE_NAME = f"{_agent_runtime_name}.DEFAULT"

print(f"Online eval config name : {ONLINE_EVAL_CONFIG_NAME}")
print(f"Monitoring log group    : {CW_LOG_GROUP}")
print(f"OTel service name       : {OTEL_SERVICE_NAME}")
print(f"Evaluators              : {list(CODE_EVAL_IDS.keys())}")
print()

online_eval_resp = cp_client.create_online_evaluation_config(
    onlineEvaluationConfigName=ONLINE_EVAL_CONFIG_NAME,
    # 세션 100% 평가, 프로덕션에서는 비용 제어를 위해 낮추세요
    rule={"samplingConfig": {"samplingPercentage": 100.0}},
    # 새 OTel span이 있는지 에이전트의 Runtime CloudWatch 로그 그룹 모니터링
    dataSourceConfig={
        "cloudWatchLogs": {
            "logGroupNames": [CW_LOG_GROUP],
            "serviceNames": [OTEL_SERVICE_NAME],
        }
    },
    # 코드 기반 evaluator와 기본 제공 evaluator를 자유롭게 조합 가능
    evaluators=[
        {"evaluatorId": CODE_EVAL_IDS["HRResponseLength"]},
        {"evaluatorId": CODE_EVAL_IDS["HRFactChecker"]},
    ],
    evaluationExecutionRoleArn=ONLINE_EVAL_ROLE_ARN,
    # enableOnCreate=True이면 생성 즉시 구성을 활성화
    enableOnCreate=True,
)

ONLINE_EVAL_CONFIG_ID = online_eval_resp["onlineEvaluationConfigId"]
ONLINE_EVAL_CONFIG_ARN = online_eval_resp.get("onlineEvaluationConfigArn", "")

print("Online eval config created:")
print(f"  ID  : {ONLINE_EVAL_CONFIG_ID}")
print(f"  ARN : {ONLINE_EVAL_CONFIG_ARN}")
print()
print("Evaluators are now LOCKED — they cannot be modified while this config is enabled.")
print("To update an evaluator: disable this config → update evaluator → re-enable.")

### 10c단계: 에이전트를 호출해 온라인 평가 트리거

이제 몇 개의 턴으로 HR Assistant를 호출합니다. 온라인 평가 구성이 활성화되어 Runtime 로그 그룹을
모니터링하므로 OTel span이 CloudWatch에 도착할 때 AgentCore가 각 세션을 자동으로 평가합니다.
명시적인 평가 API 호출은 필요하지 않습니다.

In [ ]:
# 온라인에서 자동 평가할 새 세션을 생성하도록 에이전트를 호출합니다.
# 세션별 평가를 보여 주기 위해 별도의 세션 두 개를 사용합니다.
ONLINE_SESSION_IDS = [
    f"online-eval-{uuid.uuid4()}",
    f"online-eval-{uuid.uuid4()}",
]

ONLINE_SESSION_TURNS = [
    # 세션 1: PTO 잔여 일수 + 정책 조회
    [
        "What is the PTO balance for employee EMP-001?",
        "What is the company PTO policy?",
    ],
    # 세션 2: 급여 명세서 + 복리후생
    [
        "Can you pull up the January 2026 pay stub for EMP-001?",
        "What health insurance options does the company offer?",
    ],
]

print("Invoking agent sessions (these will be auto-evaluated online)...")
for session_id, turns in zip(ONLINE_SESSION_IDS, ONLINE_SESSION_TURNS):
    print(f"\n  Session: {session_id}")
    for prompt in turns:
        print(f"    > {prompt}")
        reply = invoke_agent_simple(prompt, session_id)
        print(f"    < {reply[:100]}...")

print("\nBoth sessions invoked.")
print("AgentCore will automatically evaluate them as spans arrive in CloudWatch.")
print("Waiting 120s for CloudWatch ingestion + evaluation processing...")
time.sleep(120)
print("Ready to check online evaluation results.")

### 10d단계: 온라인 평가 결과 조회 및 표시

온라인 평가 결과는 CloudWatch Logs의 평가 결과 로그 그룹에 기록됩니다.
로그 그룹에서 세션의 평가 이벤트를 쿼리하고 점수를 표시합니다.

In [ ]:
# 온라인 평가 결과 로그 그룹은 평가 구성과 같은 리전에 있습니다
# (agent-clients 셀에서 맞춘 후 REGION = EVAL_REGION).
logs_client = boto3.client("logs", region_name=REGION)

ONLINE_EVAL_RESULTS_LOG_GROUP = "/aws/bedrock-agentcore/evaluations/online-evaluations/results/default"

look_back_ms = int(time.time() * 1000) - (30 * 60 * 1000)  # 최근 30분

print(f"Querying online eval results from: {ONLINE_EVAL_RESULTS_LOG_GROUP}")
print(f"Filtering for session IDs: {[s[:20] + '...' for s in ONLINE_SESSION_IDS]}\n")

online_results = []
try:
    paginator = logs_client.get_paginator("filter_log_events")
    for page in paginator.paginate(
        logGroupName=ONLINE_EVAL_RESULTS_LOG_GROUP,
        startTime=look_back_ms,
    ):
        for event in page.get("events", []):
            try:
                log_entry = json.loads(event["message"])
            except (json.JSONDecodeError, TypeError):
                continue

            attrs = log_entry.get("attributes", log_entry)
            session_id = attrs.get("session.id", "")

            if not any(sid == session_id for sid in ONLINE_SESSION_IDS):
                continue

            online_results.append(
                {
                    "session_id": session_id,
                    "evaluator_name": attrs.get("gen_ai.evaluation.name", ""),
                    "score": attrs.get("gen_ai.evaluation.score.value"),
                    "label": attrs.get("gen_ai.evaluation.score.label", ""),
                    "explanation": (attrs.get("gen_ai.evaluation.explanation") or "")[:120],
                }
            )

except logs_client.exceptions.ResourceNotFoundException:
    print(f"Note: Log group '{ONLINE_EVAL_RESULTS_LOG_GROUP}' not found yet.")
    print("This is normal if no sessions have been evaluated yet.")
    print("Results will appear here after AgentCore processes the first session.")

print(f"Found {len(online_results)} online evaluation result event(s).")

In [ ]:
# 온라인 평가 결과를 Markdown 표로 표시
name_by_id = {v: k for k, v in CODE_EVAL_IDS.items()}

if online_results:
    rows = [
        "| Session (truncated) | Evaluator | Score | Label | Explanation |",
        "|---|---|---|---|---|",
    ]
    for r in online_results:
        short_session = r["session_id"][:30] + "..."
        evaluator = r["evaluator_name"] or "(unknown)"
        score = str(r["score"]) if r["score"] is not None else "N/A"
        label = r["label"] or ""
        explanation = r["explanation"].replace("\n", " ")
        rows.append(f"| `{short_session}` | **{evaluator}** | {score} | {label} | {explanation} |")
    display(Markdown("### Online Evaluation Results\n\n" + "\n".join(rows)))
else:
    display(
        Markdown("""### Online Evaluation Results

> **No results yet.** Online evaluation is asynchronous — AgentCore may still be processing the
> sessions. Try re-running this cell after another 60–120 seconds.
>
> You can also check the AgentCore console or run the query below to inspect the results log group
> directly once events arrive.
""")
    )
    print(f"Log group to monitor: {ONLINE_EVAL_RESULTS_LOG_GROUP}")
    print(f"Session IDs invoked  : {ONLINE_SESSION_IDS}")

## 11단계: 정리

지속적인 비용이 발생하지 않도록 생성한 리소스를 삭제하세요.

In [ ]:
# 리소스를 정리하려면 주석을 해제하세요

# # 온라인 평가 구성 비활성화 및 삭제(잠긴 evaluator를 삭제하기 전에 비활성화해야 함)
# try:
#     cp_client.update_online_evaluation_config(
#         onlineEvaluationConfigId=ONLINE_EVAL_CONFIG_ID,
#         enableOnCreate=False,
#     )
#     print(f"Disabled online eval config: {ONLINE_EVAL_CONFIG_ID}")
# except Exception as e:
#     print(f"Could not disable online eval config: {e}")
#
# try:
#     cp_client.delete_online_evaluation_config(onlineEvaluationConfigId=ONLINE_EVAL_CONFIG_ID)
#     print(f"Deleted online eval config: {ONLINE_EVAL_CONFIG_ID}")
# except Exception as e:
#     print(f"Could not delete online eval config: {e}")

# # Lambda 함수 삭제
# for fn in ["hr-response-length", "hr-fact-checker"]:
#     try:
#         lambda_client.delete_function(FunctionName=fn)
#         print(f"Deleted Lambda: {fn}")
#     except Exception as e:
#         print(f"Could not delete {fn}: {e}")

# # evaluator 레코드 삭제(온라인 평가 구성을 삭제하거나 비활성화한 후에만 가능)
# for name, eid in CODE_EVAL_IDS.items():
#     try:
#         cp_client.delete_evaluator(evaluatorId=eid)
#         print(f"Deleted evaluator: {name} ({eid})")
#     except Exception as e:
#         print(f"Could not delete evaluator {name}: {e}")

# # agent runtime 삭제(이 Notebook에서 배포한 경우에만)
# if not _agent_loaded:
#     agentcore_control_deploy = boto3.client("bedrock-agentcore-control", region_name=REGION)
#     agentcore_control_deploy.delete_agent_runtime(agentRuntimeId=AGENT_ID)
#     print(f"Deleted agent runtime: {AGENT_ID}")

print("Cleanup skipped. Uncomment the cells above to delete resources.")

## 요약

Lambda 기반 코드 evaluator 두 개를 생성하고 다음 세 가지 방식으로 실행했습니다.

**7단계 - 온디맨드 평가(`EvaluationClient`)**: 기본 제공 LLM evaluator와 코드 기반 evaluator를
조합해 특정 프로덕션 세션을 평가했습니다.

**8단계 - `OnDemandEvaluationDatasetRunner`**: 데이터 세트 전체에서 에이전트를 자동으로 호출하고
전체 혼합 evaluator 세트로 각 시나리오의 점수를 산출했습니다.

**10단계 - 온라인 평가(`create_online_evaluation_config`)**: OTel span이 CloudWatch에 도착할 때
모든 실시간 세션의 점수를 자동으로 산출하는 지속적 평가 구성을 배포했습니다.
세션별 API 호출은 필요하지 않습니다.

| Evaluator | 유형 | 수준 | 사용 단계 |
|---|---|---|---|
| `Builtin.Correctness` | LLM | TRACE | 온디맨드(7, 8단계) |
| `Builtin.Helpfulness` | LLM | TRACE | 온디맨드(8단계) |
| `Builtin.ResponseRelevance` | LLM | TRACE | 온디맨드(8단계) |
| `HRResponseLength` | 코드 | TRACE | 온디맨드 **및** 온라인(7, 8, 10단계) |
| `HRFactChecker` | 코드 | SESSION | 온디맨드 **및** 온라인(7, 8, 10단계) |

### 코드 기반 evaluator 사용 시점

- **정확한 데이터 검증**: 응답에 특정 숫자, ID 또는 코드가 포함되어 있는지 확인
- **형식 준수**: 응답 구조, 길이 또는 형식 제약 조건 검증
- **비즈니스 규칙 적용**: LLM이 잘못 해석할 수 있는 도메인별 규칙 인코딩
- **대용량 평가**: 모든 프로덕션 세션에서 실행되는 평가 비용 절감
- **규제 요구 사항**: 특정 고지 또는 면책 조항이 항상 포함되는지 확인
- **지속적 모니터링**: 온라인 평가와 결합해 별도 개입 없는 프로덕션 품질 게이트 구성

### 온디맨드 평가와 온라인 평가 요약

| 구분 | 온디맨드 | 온라인 |
|---|---|---|
| 트리거 | 세션별 명시적 호출 | 호출할 때마다 자동 실행 |
| 설정 | `EvaluationClient.run()` 또는 `OnDemandEvaluationDatasetRunner` | `create_online_evaluation_config` 한 번 실행 |
| 코드 기반 evaluator | ✅ 지원 | ✅ 지원 |
| Evaluator 잠금 | 없음 | 구성 활성화 중 적용 |
| 적합한 용도 | CI/CD, 임시 디버깅 | 지속적인 프로덕션 모니터링 |

### 다음 단계

- 코드 기반 evaluator와 `EvaluationClient`를 결합해 특정 프로덕션 세션 평가
- 자동 회귀 테스트를 위해 CI/CD 파이프라인에 코드 기반 evaluator 추가
- 낮은 샘플링 비율(예: 10%)로 온라인 평가를 사용해 트래픽이 많은 에이전트를 비용 효율적으로 모니터링
- 에이전트 발전에 맞춰 추가 비즈니스 규칙으로 `HRFactChecker` 확장
